# PartnerCopilot v2: experimentos A, B e C no Kaggle

Importe este arquivo como um NOVO notebook; preserve o notebook e os outputs v1.
Ative GPU, Internet e o Secret HF_TOKEN. Execute em ordem em um kernel novo.

- A: revalidar as 30 saidas historicas v1 com parser-v2, sem inferencia.
- B: nova inferencia nas 30 perguntas com prompt v2 e parser estrito v1.
- C: revalidar as mesmas saidas de B com parser-v2, sem nova inferencia.
- D: retry ainda NAO implementado; este notebook nao faz tentativas adicionais de correcao.

v2 foi criada depois de ver a v1 nas mesmas 30 perguntas. Sao experimentos de
desenvolvimento, nao teste independente. Nenhum resultado v2 foi gerado localmente.
Todos os arquivos novos ficam em /kaggle/working/PartnerCopilot/v2/.

## 1. Uma GPU visível e verificação segura

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Antes de importar Torch ou inicializar CUDA.
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Ative um runtime GPU antes de continuar.')
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.device_count() == 1, 'Reinicie o kernel para aplicar CUDA_VISIBLE_DEVICES.'

## 2. Segredos e caminhos
Salve uma versão com outputs ou baixe os artefatos antes de encerrar a sessão; `/kaggle/working` não é um Drive sincronizado.

In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ.get('HF_TOKEN'), 'Configure o Secret HF_TOKEN.'
os.environ['CHROMA_PATH'] = '/kaggle/working/chroma-v2'
os.environ['DATA_EVAL_DIR'] = 'data/eval/human-v2'
os.environ['EMBEDDING_DEVICE'] = 'cpu'
os.environ['PROMPT_VERSION'] = 'partner-copilot-v2'
os.environ['PARSER_VERSION'] = 'parser-v1'  # B isola apenas a mudanca de prompt.
result_dir = Path('/kaggle/working/PartnerCopilot/v2')
result_dir.mkdir(parents=True, exist_ok=True)
output = result_dir / 'generation-kaggle-B-prompt-v2-parser-v1.jsonl'

## 3. Clonar o codigo versionado e instalar
Checkout isolado e fixo no commit com prompt/parser independentes. Nao altera o checkout v1.
Se este kernel ja executou outro notebook, reinicie antes de continuar.
Nao ha git pull automatico nem token do GitHub na URL.

In [ ]:
import subprocess, sys
EXPECTED_COMMIT = '3ec7399a451e56df02c396a16de407108d05ca73'
repo = Path('/kaggle/working/partner-copilot-v2')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoaoNordemann/partner-copilot.git', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == EXPECTED_COMMIT, 'Checkout diferente; use um diretorio novo para outro experimento.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Commit:', commit)

## A. Revalidar as saidas v1 sem GPU nem nova inferencia
A origem e o JSONL original versionado no repositorio, conferido por SHA-256.
Nao precisa reenviar os arquivos da v1. Dados reais das reviews sao usados no replay;
nenhuma nota ou metadado ficticio e fabricado. A saida e uma analise derivada separada.

In [ ]:
import json, hashlib, re
from collections import Counter
from partner_copilot.evaluation.dataset import load_eval_set
from partner_copilot.evaluation.journal import QuestionJournal
from partner_copilot.generation.output import parse_generation, parser_identity
from partner_copilot.generation.contracts import GenerationContractError, InvalidCitationError
from partner_copilot.schemas import RetrievedDoc


def replay(source_path, target_path, label):
    source_path, target_path = Path(source_path), Path(target_path)
    assert source_path.resolve() != target_path.resolve(), 'Nunca sobrescrever a origem.'
    source_bytes = source_path.read_bytes()
    source_rows = [json.loads(line) for line in source_bytes.decode('utf-8').splitlines() if line.strip()]
    ids = [f'q{i:02}' for i in range(1, 31)]
    assert [row['question_id'] for row in source_rows] == ids, 'Exige 30 perguntas completas e ordenadas.'
    assert len({row['experiment_id'] for row in source_rows}) == 1, 'Experimentos misturados.'
    _, corpus, _ = load_eval_set(Path('data/eval/human-v2'))
    reviews = {review.id: review for review in corpus}
    identity = {'analysis': label, 'new_inference': False,
                'source_sha256': hashlib.sha256(source_bytes).hexdigest(),
                'source_experiment_id': source_rows[0]['experiment_id'],
                **parser_identity('parser-v2')}
    identity['experiment_id'] = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    with QuestionJournal(target_path, identity['experiment_id'], ids, resume=True) as journal:
        for row in source_rows[len(journal.records):]:
            result = {**identity, 'question_id': row['question_id'],
                      'prompt_version': row['prompt_version'], 'deve_abster': row['deve_abster'],
                      'llm_called': row['llm_called'], 'status': 'ok', 'accepted': False,
                      'parsed': None, 'error': None}
            if row['llm_called']:
                docs = []
                for doc in row['context']:
                    review = reviews[doc['review_id']]
                    assert review.text == doc['text'], 'Contexto diferente do corpus congelado.'
                    docs.append(RetrievedDoc(review=review, score=doc['score']))
                try:
                    parsed = parse_generation(row['raw_response'], docs, parser_version='parser-v2')
                    result.update(accepted=True, parsed={'answer': parsed.answer,
                        'sources': parsed.sources, 'abstained': parsed.abstained})
                except GenerationContractError as exc:
                    result.update(status='error', error={'type': type(exc).__name__, 'message': str(exc)})
            journal.append(result)
        replay_rows = journal.records.copy()
    called = [row for row in replay_rows if row['llm_called']]
    counts = {'experiment': label, 'new_inference': False, 'completed_questions': len(replay_rows),
              'llm_outputs': len(called), 'accepted': sum(row['accepted'] for row in called),
              'rejected': sum(not row['accepted'] for row in called),
              'no_llm': sum(not row['llm_called'] for row in replay_rows),
              'errors': dict(Counter(row['error']['message'] for row in called if row['error']))}
    target_path.with_suffix('.summary.json').write_text(json.dumps(counts, indent=2), encoding='utf-8')
    print(json.dumps(counts, indent=2))
    return replay_rows

v1_source = repo / 'reports/generation-kaggle-v1/generation-kaggle-v1.jsonl'
expected_v1_hash = '23f8945599009e4f1bf454e6dd0130494ecf10ae18296ef81ccd2651c9051974'
assert hashlib.sha256(v1_source.read_bytes()).hexdigest() == expected_v1_hash
analysis_a = replay(v1_source, result_dir / 'generation-kaggle-A-parser-v2.jsonl', 'A')

## 4. Reconstruir o índice isolado
Usamos as 100 reviews do corpus human-v2, iguais às da avaliação local. Não ingerir o gold no índice de produção. Reexecutar esta célula cria outro diretório isolado.

In [ ]:
import json, uuid
from partner_copilot.config import Settings
from partner_copilot.evaluation.dataset import load_eval_set
from partner_copilot.ingestion.embedder import SentenceTransformerEmbedder
from partner_copilot.ingestion.chroma_store import ChromaStore
from partner_copilot.retrieval.retriever import Retriever
settings = Settings(prompt_version="partner-copilot-v2", parser_version="parser-v1")
settings.export_hf_env()
questions, corpus, manifest = load_eval_set(settings.data_eval_dir)
store = ChromaStore(settings.chroma_path / ('eval-' + uuid.uuid4().hex),
    collection='isolated_generation_eval', embedder=SentenceTransformerEmbedder(settings), settings=settings)
store.upsert(corpus)
assert store.count() == len(corpus) == 100
retriever = Retriever(store, settings=settings)
print('Perguntas:', len(questions), 'Documentos:', store.count())

## 5. Paridade do retrieval antes de carregar o Mistral
Comparamos ranking sem corte e resultado operacional com o relatório local, mantendo os filtros. q13 pode retornar vazio por causa do limiar. A comparação de scores é registrada; diferenças de IDs interrompem a execução para investigação.

In [ ]:
report = json.loads(Path('reports/human-v2/retrieval-comparison.json').read_text(encoding='utf-8'))
assert report['snapshot_sha256_lf'] == manifest['file_sha256_lf'], 'Snapshot diferente.'
assert settings.embedding_model.rsplit('/', 1)[-1] == report['model'].rsplit('/', 1)[-1]
assert settings.retrieval_k == 8 and settings.retrieval_min_score == report['semantic_operational_min_score']
reference = {q['id']: q for q in report['methods']['semantic']['com_filtro']['questions']}
ranker = Retriever(store, settings=settings.model_copy(update={'retrieval_min_score': 0.0}))
parity = []
for identifier in ('q01', 'q13', 'q25'):
    q = next(q for q in questions if q.id == identifier)
    filters = q.filtros.model_dump(exclude_none=True)
    ranked = ranker.retrieve(q.pergunta, **filters)
    operational = retriever.retrieve(q.pergunta, **filters)
    expected = reference[identifier]
    row = {'id': identifier, 'ranked_ids': [d.review.id for d in ranked],
           'local_ranked_ids': expected['ranked_ids'], 'scores': [d.score for d in ranked],
           'local_scores': expected['scores'], 'operational_ids': [d.review.id for d in operational],
           'local_operational_ids': expected['operational_ids']}
    row['same_ids'] = row['ranked_ids'] == row['local_ranked_ids'] and row['operational_ids'] == row['local_operational_ids']
    parity.append(row)
output.with_suffix('.retrieval-check.json').write_text(json.dumps(parity, indent=2), encoding='utf-8')
print(json.dumps(parity, indent=2))
assert all(row['same_ids'] for row in parity), 'Investigue modelo, filtros, snapshot e empates antes de continuar.'

## 6. Smoke test separado e uma unica instancia do Mistral
O teste q01 preserva a saida e qualquer erro de contrato. Um erro de formato nao
interrompe a medicao das outras perguntas. Nao altere as versoes ao ver o teste.
Falhas ao carregar o modelo continuam interrompendo a execucao.
A rodada B reutiliza essa instancia; A e C nao chamam o modelo.

In [ ]:
import hashlib
from importlib.metadata import version
from partner_copilot.generation.provider import get_llm
from partner_copilot.evaluation.generation import RecordingRetriever, RecordingLLM, run_generation
from partner_copilot.pipeline import RAGPipeline
# Mesma identidade de ambiente usada pela CLI; nenhum segredo é incluído.
source_digest = hashlib.sha256()
for source in sorted((repo / 'src/partner_copilot').rglob('*.py')):
    source_digest.update(str(source.relative_to(repo)).replace('\\', '/').encode())
    source_digest.update(source.read_text(encoding='utf-8').encode())
source_digest.update((repo / 'scripts/evaluate_generation.py').read_text(encoding='utf-8').encode())
environment = {'python': sys.version.split()[0],
    'packages': {name: version(name) for name in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'chromadb', 'sentence-transformers')},
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'torch_cuda_version': torch.version.cuda,
    'source_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'source_code_sha256': source_digest.hexdigest()}
output.with_suffix('.environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
llm = get_llm(settings)
q = next(q for q in questions if q.id == 'q01')
trace = {}
smoke = {'question_id': q.id, 'environment': environment}
try:
    answer = RAGPipeline(RecordingRetriever(retriever, trace, settings),
        RecordingLLM(lambda: llm, trace), settings=settings).ask(q.pergunta, **q.filtros.model_dump(exclude_none=True))
    smoke['answer'] = answer.model_dump()
except Exception as exc:
    smoke['error_type'] = type(exc).__name__
finally:
    smoke.update(trace)
    output.with_suffix('.smoke.json').write_text(json.dumps(smoke, ensure_ascii=False, indent=2), encoding='utf-8')
print(smoke.get('answer', {'error_type': smoke.get('error_type')}))
if 'error_type' in smoke:
    print('Smoke rejeitado pelo contrato; erro preservado. A rodada B medira os mesmos contratos sem ajustes.')

## 7. Experimento B: prompt v2, parser v1
Mesmos dados, retrieval, contexto, seed e parametros. Uma linha duravel por pergunta.
A primeira chamada de cada pergunta fica preservada; nenhum retry corretivo e feito.
Retomada exige o mesmo experimento e nao repete erros ja gravados.

In [ ]:
rows = run_generation(settings.data_eval_dir, output, settings, retriever,
    lambda: llm, environment=environment, backend='real_mistral_nf4', resume=True)
print('Tentativas salvas:', len(rows), 'Arquivo:', output)

## 8. Ambiente, métricas e revisão
Taxas finais exigem 30 tentativas. Leia pelo menos as dez respostas selecionadas, incluindo todas as negativas. IDs válidos não garantem apoio factual. A duração depende do runtime; não há benchmark real na preparação local.

In [ ]:
import runpy
helpers = runpy.run_path(str(repo / 'scripts/evaluate_generation.py'))
rows = helpers['read_results'](output)
summary = helpers['write_summary'](output, rows)
print(json.dumps(environment, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(output.with_suffix('.review.md').read_text(encoding='utf-8'))
print('Preserve o JSONL e os arquivos environment, retrieval-check, smoke, summary e review.')

## C. Parser v2 sobre as mesmas saidas de B
Nao gere as respostas novamente: este passo usa os textos brutos salvos por B.
A comparacao B versus C isola a normalizacao de underscores. Erros permanecem
registrados e abstencao continua estrita. Metricas tolerantes nao sao implementadas aqui.

In [ ]:
analysis_c = replay(output, result_dir / 'generation-kaggle-C-parser-v2.jsonl', 'C')

## D. Pendente: uma tentativa corretiva
O retry ainda nao foi implementado no projeto. Nao reexecute B para simular D:
isso nao injeta a mensagem de erro nem preserva uma segunda tentativa identificada.
Este notebook cobre A/B/C e nao publica resultados de D.

## Baixar e preservar a rodada v2
O diretorio working pode se perder ao encerrar a sessao. Baixe o ZIP ou preserve
os outputs em uma versao do Kaggle antes de sair. Arquivos de v1 nao entram neste ZIP.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
zip_path = Path('/kaggle/working/generation-kaggle-v2.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for artifact in sorted(result_dir.iterdir()):
        if artifact.is_file() and artifact.suffix != '.lock':
            archive.write(artifact, arcname=artifact.name)
display(FileLink(str(zip_path)))